# Setup

In [ ]:
from pathlib import Path
import tempfile
import pandas as pd
import numpy as np
import sys

repo_root = Path.cwd().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Final.config import PipelineConfig, default_config
from Final.paths import PROJECT_ROOT

from Final.labeling.sprint3_runner import run_sprint3_for_ptx, sprint3_results_to_frame
from Final.labeling.sprint3_standardize import load_sprint3_shrub_csv, combine_standardized_objects
from Final.labeling.object_refinement import refine_shrub_objects
from Final.labeling.transforms import transform_objects_to_als, shrub_csv_to_transform_name
from Final.labeling.alignment import align_objects_to_naip
from Final.labeling.rasterize import (
    rasterize_objects,
    resample_single_band,
    write_single_band_geotiff,
)
from Final.labeling.dedup import deduplicate_artifact_table
from Final.labeling.qa import create_overlay_figure
from Final.labeling.export import export_table
from Final.labeling.io import download_file, extract_als_metadata
from Final.labeling.manifests import (
    cleanup_path,
    list_files_with_suffix,
    site_to_remote_base,
    site_to_tif_name,
)

cfg = default_config(repo_root / 'Final' / 'artifacts')
cfg.debug = True
cfg

In [ ]:
# adjust these to your local repo if needed
FINAL_DIR = Path(PROJECT_ROOT) / "Final"
SPRINT3_BASE_DIR = Path(PROJECT_ROOT) / "Sprint 3" / "Base"
LABEL_OUTPUT_ROOT = FINAL_DIR / "labeling" / "outputs"
LABEL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# if running locally against PTX files first
PTX_BY_SITE = {
    # fill these in as you add site-specific PTX files locally
    # "calaveras-big-trees": Path(PROJECT_ROOT) / "Sprint 3" / "Base" / "CAAEU_0001_20250722_1.ptx",
}

# Run Sprint 3 (InteLLiMon)

In [ ]:
# Run only for sites where you actually have PTX locally
sprint3_results = []

for site, ptx_path in PTX_BY_SITE.items():
    result = run_sprint3_for_ptx(
        site_id=site,
        ptx_path=ptx_path,
        sprint3_base_dir=SPRINT3_BASE_DIR,
        output_root=LABEL_OUTPUT_ROOT,
        variant="revised",  # switch to "original" to compare
    )
    sprint3_results.append(result)

sprint3_results_df = sprint3_results_to_frame(sprint3_results)
sprint3_results_df

# Standardize Sprint 3 shrub outputs

In [ ]:
sprint3_dir = cfg.output.labeling_root / 'sprint3' / 'revised'
manifest_df = build_local_sprint3_manifest(sprint3_dir, site_id='calaveras-big-trees', variant='revised')
objects = standardize_sprint3_directory(sprint3_dir / 'Shrubs', site_id='calaveras-big-trees', source_version='sprint3_revised')
objects.head()

In [ ]:
refined_objects = refine_shrub_objects(objects, cfg)
refined_objects.head()
summarize_objects(refined_objects)


# Run Sprint 4

In [ ]:
def build_als_metadata_for_site(site: str, temp_dir: Path) -> list[dict]:
    als_url = f"{site_to_remote_base(cfg, site)}/{cfg.data.als_dir}"
    als_files = list_files_with_suffix(als_url, (".laz", ".las", ".copc.laz"))
    if not als_files:
        raise RuntimeError(f"No ALS files found for site '{site}' at {als_url}")

    records = []
    for entry in als_files:
        local_path = temp_dir / "als" / entry["name"]
        download_file(entry["url"], local_path)
        meta = extract_als_metadata(local_path)
        meta["source_file"] = entry["name"]
        records.append(meta)
    return records

In [ ]:
def process_one_shrub_csv_to_labels(
    *,
    site: str,
    shrub_csv_path: Path,
    source_version: str,
    naip_path: Path,
    als_meta: list[dict],
    temp_dir: Path,
    metrics_csv: Path | None = None,
    tree_inventory_csv: Path | None = None,
    fuels_raster: Path | None = None,
    dtm_raster: Path | None = None,
    chm_raster: Path | None = None,
    input_ptx: Path | None = None,
):
    transform_name = shrub_csv_to_transform_name(shrub_csv_path.name)
    transform_url = f"{site_to_remote_base(cfg, site)}/{cfg.data.transformations_dir}/{transform_name}"
    transform_local = temp_dir / "transforms" / transform_name
    download_file(transform_url, transform_local)

    objects = load_sprint3_shrub_csv(
        shrub_csv_path,
        site_id=site,
        source_version=source_version,
        label_variant="base",
        metrics_csv=metrics_csv,
        tree_inventory_csv=tree_inventory_csv,
        fuels_raster=fuels_raster,
        dtm_raster=dtm_raster,
        chm_raster=chm_raster,
        input_ptx=input_ptx,
    )

    objects = refine_shrub_objects(objects, cfg)
    objects, tile = transform_objects_to_als(objects, transform_local, als_meta)

    tile_wkt = tile.get("srs_wkt")
    if not tile_wkt:
        raise ValueError(f"ALS tile {tile.get('source_file')} is missing CRS/WKT metadata.")

    objects, grid = align_objects_to_naip(objects, naip_path, tile_wkt, cfg)
    binary, confidence, object_id = rasterize_objects(objects, grid, cfg)

    site_dir = LABEL_OUTPUT_ROOT / site
    site_dir.mkdir(parents=True, exist_ok=True)

    stem = shrub_csv_path.stem
    binary_path = site_dir / f"{stem}_mask.tif"
    confidence_path = site_dir / f"{stem}_confidence.tif"
    object_id_path = site_dir / f"{stem}_object_id.tif"
    object_table_path = site_dir / f"{stem}_objects.csv"

    write_single_band_geotiff(binary_path, binary, grid, dtype="uint8", nodata=cfg.raster.background_value)
    write_single_band_geotiff(confidence_path, confidence, grid, dtype="float32", nodata=cfg.raster.confidence_background)
    write_single_band_geotiff(object_id_path, object_id, grid, dtype="int32", nodata=0)
    export_table(objects, object_table_path)

    qa_path = LABEL_OUTPUT_ROOT / "qa" / site / f"{stem}_overlay.png"
    create_overlay_figure(naip_path, binary_path, qa_path)

    artifacts = []
    for res in cfg.raster.create_multires:
        if abs(res - grid.pixel_size_x) < 1e-6 and abs(res - grid.pixel_size_y) < 1e-6:
            multires_binary_path = binary_path
            multires_conf_path = confidence_path
        else:
            b_res, b_grid = resample_single_band(binary, grid, res)
            c_res, c_grid = resample_single_band(confidence, grid, res)
            multires_binary_path = site_dir / f"{stem}_mask_{res:g}m.tif"
            multires_conf_path = site_dir / f"{stem}_confidence_{res:g}m.tif"
            write_single_band_geotiff(multires_binary_path, b_res, b_grid, dtype="uint8", nodata=cfg.raster.background_value)
            write_single_band_geotiff(multires_conf_path, c_res, c_grid, dtype="float32", nodata=cfg.raster.confidence_background)

        artifacts.append(
            {
                "site_id": site,
                "plot_id": stem,
                "label_variant": "base",
                "resolution_m": float(res),
                "binary_mask_path": str(multires_binary_path),
                "confidence_mask_path": str(multires_conf_path),
                "object_id_raster_path": str(object_id_path),
                "object_table_path": str(object_table_path),
                "qa_overlay_path": str(qa_path),
                "n_objects": int(len(objects)),
                "n_valid_objects": int(objects["valid_object"].sum()),
                "date_token": stem,
                "source_version": source_version,
            }
        )

    return objects, pd.DataFrame(artifacts)

In [ ]:
def prepare_site_assets(site: str):
    site_base = site_to_remote_base(cfg, site)
    shrubs_dir = cfg.data.revised_shrubs_dir
    shrub_url = f"{site_base}/{shrubs_dir}"
    naip_url = f"{site_base}/{cfg.data.naip_3dep_dir}/{site_to_tif_name(site)}"

    site_temp = Path(tempfile.mkdtemp(prefix=f"label_pipeline_{site}_"))
    naip_local = site_temp / "naip" / site_to_tif_name(site)
    download_file(naip_url, naip_local)
    als_meta = build_als_metadata_for_site(site, site_temp)
    shrub_files = list_files_with_suffix(shrub_url, (".csv",))
    return site_temp, naip_local, als_meta, shrub_files

In [ ]:
TEST_SITE = cfg.sites[0]  # change as needed

site_temp, naip_local, als_meta, shrub_files = prepare_site_assets(TEST_SITE)

all_objects = []
all_artifacts = []

try:
    for csv_entry in shrub_files:
        shrub_local = site_temp / "shrubs" / csv_entry["name"]
        download_file(csv_entry["url"], shrub_local)

        objects_df, artifacts_df = process_one_shrub_csv_to_labels(
            site=TEST_SITE,
            shrub_csv_path=shrub_local,
            source_version="sprint3_revised",
            naip_path=naip_local,
            als_meta=als_meta,
            temp_dir=site_temp,
        )
        all_objects.append(objects_df)
        all_artifacts.append(artifacts_df)

finally:
    if not cfg.data.keep_temp:
        cleanup_path(site_temp)

objects_test = combine_standardized_objects(all_objects)
artifacts_test = pd.concat(all_artifacts, ignore_index=True) if all_artifacts else pd.DataFrame()

objects_test.head(), artifacts_test.head()

In [ ]:
print("Objects:", len(objects_test))
print("Valid objects:", int(objects_test["valid_object"].sum()) if len(objects_test) else 0)

display(
    objects_test[[
        "site_id", "plot_id", "object_id", "x_tls", "y_tls",
        "height_tls", "area_tls", "radius_m", "radius_source",
        "object_confidence"
    ]].head(20)
)

display(artifacts_test)

In [ ]:
artifacts_test_dedup = deduplicate_artifact_table(artifacts_test)
artifacts_test_dedup

In [ ]:
all_site_objects = []
all_site_artifacts = []

for site in cfg.sites:
    print(f"Processing {site} ...")
    site_temp, naip_local, als_meta, shrub_files = prepare_site_assets(site)
    try:
        for csv_entry in shrub_files:
            shrub_local = site_temp / "shrubs" / csv_entry["name"]
            download_file(csv_entry["url"], shrub_local)

            objects_df, artifacts_df = process_one_shrub_csv_to_labels(
                site=site,
                shrub_csv_path=shrub_local,
                source_version="sprint3_revised",
                naip_path=naip_local,
                als_meta=als_meta,
                temp_dir=site_temp,
            )
            all_site_objects.append(objects_df)
            all_site_artifacts.append(artifacts_df)
    finally:
        if not cfg.data.keep_temp:
            cleanup_path(site_temp)

objects_all = combine_standardized_objects(all_site_objects)
artifacts_all = pd.concat(all_site_artifacts, ignore_index=True) if all_site_artifacts else pd.DataFrame()
artifacts_all = deduplicate_artifact_table(artifacts_all)

print("All objects:", len(objects_all))
print("All artifact rows:", len(artifacts_all))

In [ ]:
summary_dir = LABEL_OUTPUT_ROOT / "summaries"
summary_dir.mkdir(parents=True, exist_ok=True)

objects_all.to_csv(summary_dir / "objects_all.csv", index=False)
artifacts_all.to_csv(summary_dir / "artifacts_all.csv", index=False)

print("Saved:")
print(summary_dir / "objects_all.csv")
print(summary_dir / "artifacts_all.csv")